# DataOps Patterns — Testing Pyramid, Blue-Green Deploys, Data Contracts

## Mental Model

DataOps is the discipline of treating data pipelines like production software systems.

This notebook focuses on four practical patterns:

1. **The DataOps loop** — plan → develop → test → release  
2. **The testing pyramid** — fast cheap checks at the bottom, slower broader checks at the top  
3. **Blue-green pipeline deploys** — validate new pipeline outputs before switching traffic  
4. **Data contracts** — encode table expectations as machine-checkable rules  

Citi telemetry context used throughout this notebook:

- PostgreSQL: `localhost:5432`
- Database: `de_telemetry`
- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows
- Narrative: 6,000+ API endpoints monitored for latency, error rate, and throughput; alerts escalate through severity tiers

The notebook is written as a production-style artifact for local execution against the running stack.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from dataclasses import dataclass, asdict
from pathlib import Path

try:
    import psycopg2
except ImportError as exc:
    raise RuntimeError("psycopg2 is required.") from exc

os.environ["PYTHONIOENCODING"] = "utf-8"

DBT_PATH = Path(r"C:/py_venv/proj_educate/Scripts/dbt.exe")
WORKSPACE_ROOT = Path(r"D:/Workspace/Technologies")
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

PG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

TECH_STACK = {
    "kafka": {"host": "localhost:9092", "image": "confluentinc/cp-kafka:7.6.0", "container": "citi_kafka"},
    "spark": {"version": "pyspark==3.5.4", "master": "local[*]", "JAVA_HOME": r"C:/Program Files/Java/jre1.8.0_481", "HADOOP_HOME": r"C:/hadoop"},
    "airflow": {"url": "http://localhost:8082", "image": "apache/airflow:2.8.0", "executor": "LocalExecutor", "username": "admin", "password": "admin"},
    "mlflow": {"url": "http://localhost:5000", "backend": "SQLite"},
    "dbt": {"path": str(DBT_PATH), "profiles": str(Path.home() / ".dbt" / "profiles.yml"), "project": "citi_dbt", "target": "postgres"},
    "databricks": {"host": "https://dbc-9f35a83d-b4e7.cloud.databricks.com", "warehouse_id": "b6657f31d1e7a179"},
    "gcp": {"project": "citi-de-learning", "key": r"D:/Workspace/Technologies/_setup/gcp_key.json"},
    "azure": {"subscription": "b3811436-61fc-4a3a-a6a9-deb05955076d", "az_cli": r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"},
    "aws": {"profile": "study", "region": "us-east-1", "account": "357811130281"},
}

def pg_scalar(sql: str):
    with psycopg2.connect(**PG) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            row = cur.fetchone()
            return row[0] if row else None

def pg_rows(sql: str):
    with psycopg2.connect(**PG) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall()

def run_command(cmd, cwd=None, check=True):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print(f"\n$ {printable}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        shell=isinstance(cmd, str),
        env=os.environ.copy(),
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {printable} (exit {result.returncode})")
    return result

dataset_summary = {
    "endpoints_rows": pg_scalar("select count(*) from endpoints;"),
    "metrics_rows": pg_scalar("select count(*) from metrics;"),
    "alerts_rows": pg_scalar("select count(*) from alerts;"),
}

print(json.dumps(dataset_summary, indent=2))
assert dataset_summary["endpoints_rows"] == 10000
assert dataset_summary["metrics_rows"] == 500000
assert dataset_summary["alerts_rows"] == 25000
print("Local environment validation passed.")


## DataOps Loop

The four-phase loop is:

- **Plan** — define the data contract, table shape, and release intent  
- **Develop** — change SQL, dbt models, tests, and orchestration code  
- **Test** — validate quality with dbt tests, GE checks, and direct assertions  
- **Release** — promote the validated output into the active serving layer  

### Citi mapping

- **Plan** = define the schema contract for `alerts`
- **Develop** = update a dbt model that produces an alert mart
- **Test** = run Great Expectations or row/null checks on raw alerts
- **Release** = run `dbt run` and promote validated output into the serving schema


In [2]:
dataops_loop = [
    {"phase": "plan", "citi_example": "Define schema contract for alerts severity, endpoint_id, and minimum row count."},
    {"phase": "develop", "citi_example": "Implement dbt model changes for alert marts and add singular tests."},
    {"phase": "test", "citi_example": "Run GE checkpoint or direct SQL assertions for nulls, counts, and accepted values."},
    {"phase": "release", "citi_example": "Run dbt into the promotion target and switch the active serving layer."},
]

diagram = """
+--------+     +----------+     +------+     +---------+
| PLAN   | --> | DEVELOP  | --> | TEST | --> | RELEASE |
+--------+     +----------+     +------+     +---------+
      ^                                              |
      |______________________________________________|
""".strip("\n")

print(json.dumps(dataops_loop, indent=2))
print()
print(diagram)


## Data Testing Pyramid

Why the pyramid shape matters:

- **Unit tests** are cheapest and fastest, so you run many of them.
- **Integration tests** are broader and slower, so you run fewer.
- **End-to-end tests** prove the full pipeline path, but they are the most expensive.

So the base should be large, and the top should be narrow.

This notebook models three layers:

1. **Unit** — dbt singular tests on `mart_alert_summary`
2. **Integration** — raw alerts validation with row count and null checks
3. **End-to-end** — dbt run followed by `mart_alert_summary` row count > 0


In [3]:
class DataTestingPyramid:
    def __init__(self, dbt_path: Path, db_project_candidates=None):
        self.dbt_path = dbt_path
        self.db_project_candidates = db_project_candidates or [
            WORKSPACE_ROOT / "citi_dbt",
            WORKSPACE_ROOT / "dbt" / "citi_dbt",
            WORKSPACE_ROOT,
        ]

    def _find_project_dir(self):
        for path in self.db_project_candidates:
            if (path / "dbt_project.yml").exists():
                return path
        return None

    def unit_layer(self):
        project_dir = self._find_project_dir()
        if project_dir is None or not self.dbt_path.exists():
            return {
                "layer": "unit",
                "passed": False,
                "detail": "dbt project or dbt executable not found; cannot run singular tests on mart_alert_summary.",
            }

        try:
            result = run_command(
                [str(self.dbt_path), "test", "--project-dir", str(project_dir), "--select", "mart_alert_summary"],
                check=False,
            )
            return {
                "layer": "unit",
                "passed": result.returncode == 0,
                "detail": result.stdout[-1000:] if result.stdout else "dbt test completed with no stdout",
                "returncode": result.returncode,
            }
        except Exception as exc:
            return {"layer": "unit", "passed": False, "detail": str(exc)}

    def integration_layer(self):
        row_count = pg_scalar("select count(*) from alerts;")
        null_endpoint_id = pg_scalar("select count(*) from alerts where endpoint_id is null;")
        null_severity = pg_scalar("select count(*) from alerts where severity is null;")
        passed = (row_count is not None and row_count > 0 and null_endpoint_id == 0 and null_severity == 0)
        return {
            "layer": "integration",
            "passed": passed,
            "detail": {
                "alerts_row_count": row_count,
                "null_endpoint_id": null_endpoint_id,
                "null_severity": null_severity,
            },
        }

    def end_to_end_layer(self):
        project_dir = self._find_project_dir()
        if project_dir is None or not self.dbt_path.exists():
            return {
                "layer": "end_to_end",
                "passed": False,
                "detail": "dbt project or dbt executable not found; cannot run dbt build path.",
            }

        try:
            dbt_result = run_command(
                [str(self.dbt_path), "run", "--project-dir", str(project_dir), "--select", "mart_alert_summary"],
                check=False,
            )
            mart_exists = pg_scalar("""
                select count(*) 
                from information_schema.tables 
                where table_schema='public' and table_name='mart_alert_summary';
            """)
            mart_row_count = 0
            if mart_exists:
                mart_row_count = pg_scalar("select count(*) from public.mart_alert_summary;")
            passed = dbt_result.returncode == 0 and mart_exists == 1 and mart_row_count > 0
            return {
                "layer": "end_to_end",
                "passed": passed,
                "detail": {
                    "dbt_returncode": dbt_result.returncode,
                    "mart_exists": mart_exists,
                    "mart_row_count": mart_row_count,
                },
            }
        except Exception as exc:
            return {"layer": "end_to_end", "passed": False, "detail": str(exc)}

    def run_all(self):
        return [self.unit_layer(), self.integration_layer(), self.end_to_end_layer()]

pyramid = DataTestingPyramid(DBT_PATH)
pyramid_results = pyramid.run_all()

for result in pyramid_results:
    print(f"{result['layer']}: {'PASS' if result['passed'] else 'FAIL'}")
    print(json.dumps(result, indent=2, default=str))
    print("-" * 80)


## Blue-Green Pipeline Deploy

Blue-green deploys reduce promotion risk.

Pattern:

- build new output in an inactive target
- validate it
- switch the active alias only after validation passes

For pipelines, that usually means:

- **blue** = current serving schema
- **green** = candidate schema
- **active view / alias** = what consumers read

This pattern avoids downtime because readers keep hitting the active alias while the candidate version is being prepared and checked.


In [4]:
with psycopg2.connect(**PG) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("create schema if not exists citi_blue;")
        cur.execute("create schema if not exists citi_green;")

        cur.execute("drop table if exists citi_blue.alerts_summary;")
        cur.execute("drop table if exists citi_green.alerts_summary;")

        summary_sql = """
        create table {schema}.alerts_summary as
        select
            endpoint_id,
            count(*) as alert_count,
            max(created_at) as latest_alert_at
        from public.alerts
        group by endpoint_id;
        """
        cur.execute(summary_sql.format(schema="citi_blue"))
        cur.execute(summary_sql.format(schema="citi_green"))

        green_count = pg_scalar("select count(*) from citi_green.alerts_summary;")
        validation_passed = green_count is not None and green_count > 0

        cur.execute("drop view if exists public.citi_active;")
        if validation_passed:
            cur.execute("""
                create view public.citi_active as
                select * from citi_green.alerts_summary;
            """)
            active_target = "citi_green.alerts_summary"
        else:
            cur.execute("""
                create view public.citi_active as
                select * from citi_blue.alerts_summary;
            """)
            active_target = "citi_blue.alerts_summary"

deployment_report = {
    "blue_row_count": pg_scalar("select count(*) from citi_blue.alerts_summary;"),
    "green_row_count": pg_scalar("select count(*) from citi_green.alerts_summary;"),
    "validation_passed": validation_passed,
    "active_target": active_target,
    "active_view_row_count": pg_scalar("select count(*) from public.citi_active;"),
}

print(json.dumps(deployment_report, indent=2))


## Data Contract Testing

A data contract makes pipeline expectations explicit.

Here we encode:

- table name
- required columns
- max allowed null percentage
- minimum row count

That gives us a reusable, machine-checkable validation layer that can sit in CI or release workflows.


In [5]:
@dataclass
class DataContract:
    table_name: str
    required_columns: list[str]
    max_null_pct: dict[str, float]
    min_row_count: int

def get_table_columns(table_name: str):
    rows = pg_rows(f"""
        select column_name
        from information_schema.columns
        where table_schema = 'public' and table_name = '{table_name}'
        order by ordinal_position;
    """)
    return [r[0] for r in rows]

def validate_contract(contract: DataContract):
    row_count = pg_scalar(f"select count(*) from public.{contract.table_name};")
    columns = get_table_columns(contract.table_name)
    missing_columns = [c for c in contract.required_columns if c not in columns]

    null_checks = {}
    for col, max_pct in contract.max_null_pct.items():
        null_count = pg_scalar(f"select count(*) from public.{contract.table_name} where {col} is null;")
        pct = (null_count / row_count * 100.0) if row_count else 100.0
        null_checks[col] = {
            "null_count": null_count,
            "null_pct": round(pct, 4),
            "max_allowed_pct": max_pct,
            "passed": pct <= max_pct,
        }

    passed = (
        row_count >= contract.min_row_count
        and not missing_columns
        and all(v["passed"] for v in null_checks.values())
    )

    return {
        "contract": asdict(contract),
        "row_count": row_count,
        "missing_columns": missing_columns,
        "null_checks": null_checks,
        "passed": passed,
    }

alerts_contract = DataContract(
    table_name="alerts",
    required_columns=["alert_id", "endpoint_id", "severity", "message", "created_at"],
    max_null_pct={"endpoint_id": 0.0, "severity": 0.0},
    min_row_count=20000,
)

endpoints_contract = DataContract(
    table_name="endpoints",
    required_columns=["endpoint_id", "name", "region", "status", "category"],
    max_null_pct={"endpoint_id": 0.0, "name": 0.0},
    min_row_count=10000,
)

contract_reports = [
    validate_contract(alerts_contract),
    validate_contract(endpoints_contract),
]

print(json.dumps(contract_reports, indent=2, default=str))


## Schema Evolution Safety

A safe additive migration usually means:

- add a nullable column
- keep old readers working
- backfill later if needed
- validate contracts and tests before relying on the new field

Breaking changes are different:

- rename a required column
- drop a required column
- change a data type incompatibly

Those should be blocked or coordinated because downstream models and contracts can fail immediately.


In [6]:
with psycopg2.connect(**PG) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            alter table public.alerts
            add column if not exists source_system varchar(100);
        """)

schema_after = get_table_columns("alerts")
assert "source_system" in schema_after

project_candidates = [
    WORKSPACE_ROOT / "citi_dbt",
    WORKSPACE_ROOT / "dbt" / "citi_dbt",
    WORKSPACE_ROOT,
]
dbt_project_dir = next((p for p in project_candidates if (p / "dbt_project.yml").exists()), None)

dbt_test_report = {
    "dbt_project_found": dbt_project_dir is not None,
    "dbt_executable_found": DBT_PATH.exists(),
    "dbt_test_ran": False,
    "dbt_test_returncode": None,
}

if dbt_project_dir is not None and DBT_PATH.exists():
    result = run_command(
        [str(DBT_PATH), "test", "--project-dir", str(dbt_project_dir)],
        check=False,
    )
    dbt_test_report["dbt_test_ran"] = True
    dbt_test_report["dbt_test_returncode"] = result.returncode

schema_safety_notes = {
    "additive_change": "Adding nullable column source_system is usually safe because existing reads still work.",
    "breaking_change_examples": [
        "rename severity to sev_level",
        "change endpoint_id from int to text without coordinated migration",
        "drop created_at while downstream models still select it",
    ],
    "why_contracts_help": "Contracts fail early when required columns disappear, row counts crater, or null percentages exceed policy.",
}

print(json.dumps(dbt_test_report, indent=2))
print(json.dumps(schema_safety_notes, indent=2))


## What Just Happened

DataOps closes the gap between data engineering and software engineering.

- The **testing pyramid** gives fast feedback at low cost.
- **Blue-green deploys** make pipeline promotions reversible and low risk.
- **Data contracts** make schema expectations explicit and machine-checkable.

In a Citi-style alerting platform spanning 6,000+ monitored endpoints, those patterns are what let teams ship safely instead of hoping production will be forgiving.
